[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_06_Multi_Agent_Systems.ipynb)

# 🤝 Lesson 6: Multi-Agent Systems
### *Your AI Learning Journey — Lesson 6 of 9*

---

## Where We Are

So far you've built:
- **Lesson 1:** Raw LLM calls (stateless API)
- **Lesson 2:** Prompt engineering (getting the LLM to think well)
- **Lesson 3:** Tool use (giving the agent hands)
- **Lesson 4:** The ReAct loop (your first real agent)
- **Lesson 5:** Agent memory (making it remember across turns)

Today we go **multi-agent**. Instead of one smart agent, we build *systems* where multiple specialized agents work together — like a team of people with different roles.

---

## What You'll Learn

1. **Why multi-agent?** — the limits of a single agent, and what you gain
2. **Orchestrators & Subagents** — the manager/worker pattern
3. **Handoff patterns** — how agents pass work to each other
4. **Sequential pipeline** — Agent A → Agent B → Agent C
5. **Parallel fan-out** — one orchestrator spawns many workers simultaneously
6. **Critic/Reviewer pattern** — one agent writes, another reviews
7. **Build it:** A real multi-agent research pipeline

---

## The Core Mental Model

Think of a software company:
- A **Project Manager** breaks down requirements and assigns tasks
- **Developers** implement specific pieces
- A **QA Engineer** reviews the output
- The PM synthesizes everything into a final deliverable

Multi-agent systems work exactly like this. The PM is the **orchestrator**. The developers and QA are **subagents**.

```
User Request
     │
     ▼
┌─────────────┐
│ ORCHESTRATOR│  ← The "brain" — plans, delegates, synthesizes
└──────┬──────┘
       │  delegates tasks
  ┌────┴────┬─────────┐
  ▼         ▼         ▼
Agent A   Agent B   Agent C   ← Specialists — each does one thing well
(search)  (analyze) (write)
  │         │         │
  └────┬────┴─────────┘
       │  results flow back
       ▼
┌─────────────┐
│ ORCHESTRATOR│  ← Synthesizes results
└─────────────┘
       │
       ▼
  Final Answer
```

---
## 🔧 Setup

**API Key Setup (one-time):**
1. Click the 🔑 key icon in Colab's left sidebar → "Secrets"
2. Add a secret named `ANTHROPIC_API_KEY` with your key from https://console.anthropic.com
3. Toggle "Notebook access" ON

Then run the cell below.

In [ ]:
!pip install anthropic -q

import anthropic
import json
import time
from typing import Any

# Load API key from Colab Secrets
try:
    from google.colab import userdata
    api_key = userdata.get('ANTHROPIC_API_KEY')
    print("✅ API key loaded from Colab Secrets")
except Exception:
    import os
    api_key = os.environ.get('ANTHROPIC_API_KEY', 'your-api-key-here')
    print("⚠️  Using environment variable — add key to Colab Secrets for best practice")

client = anthropic.Anthropic(api_key=api_key)
print("🚀 Anthropic client ready!")

---
## Part 1: Why Multi-Agent?

### The limits of a single agent

A single agent with tools works great for many tasks. But it hits walls:

| Problem | What Happens |
|---|---|
| **Context window limits** | A single agent can only hold ~100-200k tokens in memory. Complex tasks overflow it. |
| **Conflicting skills** | One agent can't be simultaneously an expert researcher, expert writer, AND expert fact-checker |
| **Parallelism** | A single agent is serial — it can't do 5 things at once |
| **Reliability** | One failure cascades — there's no one to catch mistakes |
| **Prompt complexity** | One agent with 20 tools and a 2000-line system prompt becomes unpredictable |

### What multi-agent buys you

- **Specialization:** Each agent has a focused system prompt and relevant tools only
- **Parallelism:** Fan out work, get results faster
- **Error checking:** Agent B can review Agent A's work
- **Longer workflows:** Chain agents so no single context window overflows
- **Resilience:** Retry individual agents without restarting the whole pipeline

### The key insight: agents are just functions

An agent is really just:
```python
def agent(task: str, context: dict) -> str:
    # uses LLM + tools to produce output
    return result
```

Once you think of agents as functions, composing them into systems is natural — just like calling functions from other functions.

---
## Part 2: Pattern 1 — Sequential Pipeline

The simplest multi-agent pattern: a chain. Each agent's output becomes the next agent's input.

```
Topic → [Researcher] → raw_notes → [Writer] → draft → [Editor] → final
```

**When to use:** When each step needs the full output of the previous step, and order matters.

Let's build a 3-agent content pipeline:

In [ ]:
# ─────────────────────────────────────────────
# Helper: call one agent with a system prompt and task
# ─────────────────────────────────────────────
def run_agent(name: str, system_prompt: str, task: str, model: str = "claude-haiku-4-5-20251001") -> str:
    """Run a single agent and return its text output."""
    print(f"\n🤖 [{name}] Working...")
    
    response = client.messages.create(
        model=model,
        max_tokens=1024,
        system=system_prompt,
        messages=[{"role": "user", "content": task}]
    )
    
    result = response.content[0].text
    print(f"✅ [{name}] Done ({len(result)} chars)")
    return result

print("Helper function defined ✓")

In [ ]:
# ─────────────────────────────────────────────
# SEQUENTIAL PIPELINE: Researcher → Writer → Editor
# ─────────────────────────────────────────────

TOPIC = "Why AI Agents fail in production"
print(f"🎯 Topic: {TOPIC}")
print("=" * 60)

# ── Step 1: Researcher Agent ──
researcher_system = """You are a research specialist. Given a topic, produce 
bullet-point research notes covering: key concepts, common problems, 
important nuances, and real-world examples. Be factual and thorough. 
Output raw notes, not polished prose."""

research_notes = run_agent(
    name="Researcher",
    system_prompt=researcher_system,
    task=f"Research this topic thoroughly: {TOPIC}"
)

# ── Step 2: Writer Agent (receives researcher's output) ──
writer_system = """You are a technical writer. Given research notes, write a 
clear, engaging 3-paragraph blog post section. Write for software engineers 
who are new to AI. Use concrete examples. No jargon without explanation."""

draft = run_agent(
    name="Writer",
    system_prompt=writer_system,
    task=f"Write a blog post section based on these research notes:\n\n{research_notes}"
)

# ── Step 3: Editor Agent (receives writer's output) ──
editor_system = """You are a senior editor. Given a draft, improve it by:
1. Tightening wordy sentences (aim for clarity)
2. Ensuring the opening hook is strong
3. Making sure it ends with a clear takeaway
Return ONLY the improved final text, no editorial comments."""

final_article = run_agent(
    name="Editor",
    system_prompt=editor_system,
    task=f"Edit and improve this draft:\n\n{draft}"
)

print("\n" + "=" * 60)
print("📄 FINAL ARTICLE")
print("=" * 60)
print(final_article)

### 🔍 What just happened?

Three separate LLM calls, each with a **focused system prompt**. The output of each became the input of the next. Notice:

- The **Researcher** doesn't know it's writing for a blog. It just researches.
- The **Writer** doesn't know there will be an editor. It just writes.
- The **Editor** doesn't know how the research was done. It just edits.

This **separation of concerns** is the key to why multi-agent works. Each agent can be optimized independently.

💡 **EXPERIMENT:** Try changing `TOPIC` to something in your domain. Also try switching `model` to `claude-opus-4-6` for the editor only and see if quality improves.

---
## Part 3: Pattern 2 — Orchestrator with Subagents

The orchestrator pattern adds a **coordinator** that decides *which* agent to call and *in what order*, based on the task at hand. This is more flexible than a fixed pipeline — the orchestrator can adapt.

```
                  ┌─────────────────┐
User ──────────►  │  ORCHESTRATOR   │
                  │  (plans & calls)│
                  └────────┬────────┘
                           │
             ┌─────────────┼─────────────┐
             ▼             ▼             ▼
      [Subagent A]  [Subagent B]  [Subagent C]
      (summarizer) (fact-checker) (formatter)
             │             │             │
             └─────────────┴─────────────┘
                           │
                  ┌────────▼────────┐
                  │  ORCHESTRATOR   │  synthesizes
                  └─────────────────┘
```

The orchestrator uses **tools that are actually agents**. This is a powerful pattern — from the orchestrator's perspective, calling a subagent looks identical to calling any other tool.

In [ ]:
# ─────────────────────────────────────────────
# ORCHESTRATOR PATTERN
# The orchestrator has "tools" that are actually calls to subagents
# ─────────────────────────────────────────────

# Define our subagent functions (each is a specialized mini-agent)
def summarize_agent(text: str) -> str:
    """Subagent: Summarizes a piece of text into 3 bullet points."""
    return run_agent(
        name="Summarizer",
        system_prompt="You are a summarization expert. Condense the given text into exactly 3 concise bullet points. Each bullet starts with '• '. No preamble.",
        task=text
    )

def critique_agent(text: str) -> str:
    """Subagent: Finds weaknesses and gaps in an argument."""
    return run_agent(
        name="Critic",
        system_prompt="You are a critical thinker. Given a piece of text, identify 3 weaknesses, gaps, or counterarguments. Be specific and constructive. Format as numbered list.",
        task=text
    )

def simplify_agent(text: str, audience: str = "a 10-year-old") -> str:
    """Subagent: Simplifies complex text for a given audience."""
    return run_agent(
        name="Simplifier",
        system_prompt=f"You are an expert at explaining complex topics simply. Rewrite the given text so {audience} could understand it. Use analogies and simple words.",
        task=text
    )

# ── Tool definitions for the orchestrator ──
# These look like normal tool definitions, but they call Python functions 
# that themselves call LLMs!
ORCHESTRATOR_TOOLS = [
    {
        "name": "summarize",
        "description": "Summarize a piece of text into 3 concise bullet points.",
        "input_schema": {
            "type": "object",
            "properties": {
                "text": {"type": "string", "description": "The text to summarize"}
            },
            "required": ["text"]
        }
    },
    {
        "name": "critique",
        "description": "Find weaknesses, gaps, or counterarguments in a piece of text.",
        "input_schema": {
            "type": "object",
            "properties": {
                "text": {"type": "string", "description": "The text to critique"}
            },
            "required": ["text"]
        }
    },
    {
        "name": "simplify",
        "description": "Rewrite complex text in simple language for a non-expert audience.",
        "input_schema": {
            "type": "object",
            "properties": {
                "text": {"type": "string", "description": "The text to simplify"},
                "audience": {"type": "string", "description": "Who the simplified version is for (e.g., 'a 10-year-old', 'a non-technical manager')"}
            },
            "required": ["text"]
        }
    }
]

# Tool dispatch: when orchestrator calls a tool, route to the right subagent
def dispatch_tool(tool_name: str, tool_input: dict) -> str:
    if tool_name == "summarize":
        return summarize_agent(tool_input["text"])
    elif tool_name == "critique":
        return critique_agent(tool_input["text"])
    elif tool_name == "simplify":
        audience = tool_input.get("audience", "a non-technical person")
        return simplify_agent(tool_input["text"], audience)
    else:
        return f"Unknown tool: {tool_name}"

print("Subagents and tool definitions ready ✓")

In [ ]:
# ─────────────────────────────────────────────
# The Orchestrator — runs the agent loop,
# but its "tools" are actually subagent calls!
# ─────────────────────────────────────────────

ORCHESTRATOR_SYSTEM = """You are a content analysis orchestrator. When given a piece of text and a goal, 
you use your available tools to analyze it comprehensively. 
Use multiple tools when appropriate. After collecting all results, synthesize them into a final response."""

SAMPLE_TEXT = """
Large Language Models (LLMs) are revolutionizing software development. They can write code faster 
than most humans, debug complex systems, and explain technical concepts. Many companies are now 
replacing junior developers with AI tools. The future of programming is prompt engineering, 
not traditional coding.
"""

USER_GOAL = f"""Analyze this text for me: summarize it, find its weaknesses, 
and create a version I could share with my non-technical CEO.

TEXT:
{SAMPLE_TEXT}"""

def run_orchestrator(user_message: str) -> str:
    """Run the orchestrator with its subagent tools."""
    messages = [{"role": "user", "content": user_message}]
    
    print("\n🧠 ORCHESTRATOR starting...")
    print("=" * 60)
    
    while True:
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=2048,
            system=ORCHESTRATOR_SYSTEM,
            tools=ORCHESTRATOR_TOOLS,
            messages=messages
        )
        
        # If the orchestrator is done (no more tool calls), return final answer
        if response.stop_reason == "end_turn":
            final = next(b.text for b in response.content if hasattr(b, 'text'))
            print("\n✅ ORCHESTRATOR synthesizing final answer")
            return final
        
        # Process tool calls (which are actually subagent calls!)
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                print(f"\n📞 Orchestrator calls subagent: [{block.name}]")
                print(f"   Input: {json.dumps(block.input)[:80]}..." 
                      if len(json.dumps(block.input)) > 80 
                      else f"   Input: {json.dumps(block.input)}")
                
                # THIS is the key line: calling the subagent
                result = dispatch_tool(block.name, block.input)
                
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": result
                })
        
        # Feed results back to orchestrator
        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": tool_results})

final_synthesis = run_orchestrator(USER_GOAL)

print("\n" + "=" * 60)
print("🎯 ORCHESTRATOR FINAL SYNTHESIS")
print("=" * 60)
print(final_synthesis)

### 🔍 Key Insight: The Orchestrator Doesn't Know It's Calling Other LLMs

From the orchestrator's perspective, it used three tools — `summarize`, `critique`, `simplify`. Internally, each of those called a completely separate LLM. This is the beauty of the pattern:

- **Composable:** Swap any subagent with a different implementation (rule-based, different model, API call)
- **Observable:** You can log and debug each subagent independently
- **Scalable:** Move subagents to separate services if they become complex

💡 **EXPERIMENT:** Add a 4th subagent tool called `translate` that translates text to a language of your choice. Register it in `ORCHESTRATOR_TOOLS` and `dispatch_tool`, then change the user goal to request a translation.

---
## Part 4: Pattern 3 — Parallel Fan-Out

Sometimes you need the same task done in parallel — multiple agents working simultaneously. This is where multi-agent truly shines for **speed**.

```
                 ┌───────────────┐
                 │ ORCHESTRATOR  │
                 └───────┬───────┘
                         │ splits into N tasks
          ┌──────────────┼──────────────┐
          ▼              ▼              ▼
   [Agent: Python]  [Agent: JS]  [Agent: Go]    ← all run at the same time!
          │              │              │
          └──────────────┼──────────────┘
                         │ results collected
                 ┌───────▼───────┐
                 │ ORCHESTRATOR  │ combines
                 └───────────────┘
```

We'll use Python's `concurrent.futures` to actually run multiple LLM calls in parallel.

In [ ]:
import concurrent.futures
import time

# ─────────────────────────────────────────────
# PARALLEL FAN-OUT
# Ask multiple specialist agents to answer the same question
# from different perspectives, simultaneously
# ─────────────────────────────────────────────

QUESTION = "What are the biggest risks of using AI agents in a production system?"

# Each specialist sees the question through their own lens
SPECIALISTS = [
    {
        "name": "Security Engineer",
        "system": "You are a senior security engineer. Answer from a security and threat-modeling perspective. Be specific about attack vectors and mitigations. 3-4 sentences."
    },
    {
        "name": "SRE / Reliability",
        "system": "You are a Site Reliability Engineer. Answer from a reliability, observability, and operational perspective. Focus on failure modes and monitoring. 3-4 sentences."
    },
    {
        "name": "Product Manager",
        "system": "You are an experienced product manager. Answer from a user trust, business risk, and product quality perspective. Think about what users experience when things go wrong. 3-4 sentences."
    },
    {
        "name": "ML Engineer",
        "system": "You are an ML engineer. Answer from a model behavior, hallucination, and evaluation perspective. Focus on technical AI-specific risks. 3-4 sentences."
    }
]

def call_specialist(specialist: dict, question: str) -> dict:
    """Call one specialist agent and return their perspective."""
    result = run_agent(
        name=specialist["name"],
        system_prompt=specialist["system"],
        task=question
    )
    return {"name": specialist["name"], "perspective": result}

# ── Fan-out: run all specialists IN PARALLEL ──
print(f"🎯 Question: {QUESTION}")
print(f"🚀 Spinning up {len(SPECIALISTS)} specialist agents in parallel...")
print("=" * 60)

start = time.time()

with concurrent.futures.ThreadPoolExecutor(max_workers=len(SPECIALISTS)) as executor:
    futures = [
        executor.submit(call_specialist, specialist, QUESTION)
        for specialist in SPECIALISTS
    ]
    perspectives = [f.result() for f in concurrent.futures.as_completed(futures)]

elapsed = time.time() - start
print(f"\n⚡ All {len(perspectives)} agents finished in {elapsed:.1f}s (parallel!)")

# ── Fan-in: orchestrator synthesizes ──
print("\n🧠 Orchestrator synthesizing perspectives...")

perspectives_text = "\n\n".join(
    f"**{p['name']}:**\n{p['perspective']}" for p in perspectives
)

synthesis = run_agent(
    name="Synthesis Orchestrator",
    system_prompt="""You are a senior engineering leader. Given multiple expert perspectives on a topic, 
    synthesize them into a coherent summary. Identify common themes, unique insights from each perspective, 
    and produce a prioritized list of 5 key risks. Format clearly.""",
    task=f"Question: {QUESTION}\n\nExpert Perspectives:\n{perspectives_text}"
)

print("\n" + "=" * 60)
print("🎯 SYNTHESIZED ANSWER FROM ALL SPECIALISTS")
print("=" * 60)
print(synthesis)

### ⚡ Sequential vs Parallel: The Speed Difference

If each LLM call takes ~3 seconds and you have 4 specialists:
- **Sequential:** 4 × 3s = **12 seconds**
- **Parallel:** max(3s, 3s, 3s, 3s) = **~3 seconds**

That's a **4x speedup** for free, just by using threads. For 20 agents, it's 20x faster.

💡 **EXPERIMENT:** Add a 5th specialist — perhaps a "Legal/Compliance" perspective. Add it to `SPECIALISTS` and re-run. Notice the parallel execution time barely increases!

---
## Part 5: Pattern 4 — Critic/Reviewer (Self-Improving Loop)

One of the most powerful patterns: **Agent A writes, Agent B critiques, Agent A revises**. This loop can dramatically improve output quality.

```
Task → [Generator] → Draft v1 → [Critic] → Feedback
                        ↑                       │
                        └───────────────────────┘
                            (until good enough)
```

This mirrors how humans work: write a draft, get feedback, revise. Studies show LLMs improve significantly with even one round of critique.

In [ ]:
# ─────────────────────────────────────────────
# CRITIC / REVIEWER LOOP
# Generator produces code, Critic reviews it,
# Generator revises — up to N rounds
# ─────────────────────────────────────────────

GENERATOR_SYSTEM = """You are a Python developer. When given a coding task, write clean Python code.
When given feedback, revise your code to address all issues. Return ONLY the code, no explanation."""

CRITIC_SYSTEM = """You are a senior Python code reviewer. Review code for:
1. Correctness (does it do what's asked?)
2. Edge cases (what inputs would break it?)
3. Pythonic style (is it idiomatic?)
4. Missing error handling

Rate the code PASS or NEEDS_REVISION.
If NEEDS_REVISION, provide specific, actionable feedback.
Format your response as:
VERDICT: PASS or NEEDS_REVISION
ISSUES: (list issues if NEEDS_REVISION)"""

CODING_TASK = """
Write a Python function `safe_divide(numbers: list, divisor: float) -> list` that:
- Divides each number in the list by the divisor
- Returns the results as a list
- Handles edge cases gracefully
"""

def critic_loop(task: str, max_rounds: int = 3) -> tuple[str, int]:
    """Run generator → critic → revise loop. Returns (final_code, rounds_taken)."""
    
    # ── Round 0: Generate initial version ──
    print(f"\n📝 TASK: {task.strip()[:80]}...")
    print("=" * 60)
    
    current_code = run_agent(
        name="Generator v1",
        system_prompt=GENERATOR_SYSTEM,
        task=task
    )
    print(f"\nInitial code:\n{current_code}")
    
    # ── Critic loop ──
    for round_num in range(1, max_rounds + 1):
        print(f"\n{'─'*40}")
        print(f"🔍 CRITIC REVIEW — Round {round_num}")
        print(f"{'─'*40}")
        
        critique = run_agent(
            name=f"Critic Round {round_num}",
            system_prompt=CRITIC_SYSTEM,
            task=f"Review this Python code:\n```python\n{current_code}\n```"
        )
        print(f"\nCritique:\n{critique}")
        
        # Check verdict
        if "VERDICT: PASS" in critique:
            print(f"\n🎉 PASSED after {round_num} review round(s)!")
            return current_code, round_num
        
        # Extract issues and revise
        print(f"\n✏️  Generator revising (round {round_num})...")
        current_code = run_agent(
            name=f"Generator Revision {round_num}",
            system_prompt=GENERATOR_SYSTEM,
            task=f"Original task: {task}\n\nYour previous code:\n```python\n{current_code}\n```\n\nReviewer feedback:\n{critique}\n\nRevise the code to fix all issues."
        )
        print(f"\nRevised code:\n{current_code}")
    
    print(f"\n⚠️  Reached max rounds ({max_rounds}). Returning best version.")
    return current_code, max_rounds

# Run the loop!
final_code, rounds = critic_loop(CODING_TASK, max_rounds=3)

print("\n" + "=" * 60)
print(f"✅ FINAL CODE (after {rounds} review round(s))")
print("=" * 60)
print(final_code)

### 🔍 Why Critique Works

LLMs are better at *evaluating* than *generating* on the first try. Critique taps into a different "mode" of reasoning — it's easier to spot a bug in code than to write perfect code cold.

This pattern is used in production by:
- **Constitutional AI** (Anthropic's technique for making Claude safer)
- **AlphaCode** (Google's coding model)
- Many commercial coding assistants

💡 **EXPERIMENT:** Change `CODING_TASK` to ask for something intentionally tricky — like a recursive function or one that handles Unicode. Watch how the critic catches issues the generator missed on the first try.

---
## Part 6: Putting It All Together — A Real Research Pipeline

Let's combine what we've learned into a real-world multi-agent pipeline that a developer would actually build:

**Goal:** Given a software engineering topic, produce a structured analysis report

**Architecture:**
```
Topic
  │
  ▼
[Planner] → breaks topic into N sub-questions
  │
  ├──► [Researcher 1] ──┐
  ├──► [Researcher 2] ──┤  PARALLEL
  └──► [Researcher N] ──┘
              │
  [Synthesizer] → combines into draft report
              │
  [Critic] → validates and rates
              │
         Final Report
```

In [ ]:
import json

# ─────────────────────────────────────────────
# FULL RESEARCH PIPELINE
# Planner → Parallel Researchers → Synthesizer → Critic
# ─────────────────────────────────────────────

RESEARCH_TOPIC = "How to design reliable AI agent systems"

print(f"🔬 Research topic: {RESEARCH_TOPIC}")
print("=" * 60)

# ── STEP 1: Planner decomposes the topic ──
print("\n📋 STEP 1: Planner breaks down topic...")

planner_response = run_agent(
    name="Planner",
    system_prompt="""You are a research planner. Given a broad topic, break it into exactly 3 focused sub-questions 
    that together cover the topic comprehensively. Output ONLY a JSON array of 3 strings, e.g.:
    ["question 1", "question 2", "question 3"]""",
    task=f"Break down this research topic into 3 sub-questions: {RESEARCH_TOPIC}"
)

# Parse the sub-questions
try:
    # Find JSON array in the response
    import re
    match = re.search(r'\[.*?\]', planner_response, re.DOTALL)
    sub_questions = json.loads(match.group() if match else planner_response)
except:
    # Fallback if parsing fails
    sub_questions = [
        "What are the common failure modes of AI agents?",
        "What observability and monitoring practices work best for AI agents?",
        "How do you test and evaluate AI agent behavior reliably?"
    ]

print(f"\nSub-questions identified:")
for i, q in enumerate(sub_questions, 1):
    print(f"  {i}. {q}")

# ── STEP 2: Parallel research on each sub-question ──
print(f"\n⚡ STEP 2: Running {len(sub_questions)} researchers in parallel...")

def research_question(question: str) -> dict:
    result = run_agent(
        name=f"Researcher",
        system_prompt="""You are a senior software architect. Answer the given question with:
        - 2-3 key insights supported by reasoning
        - 1 concrete, actionable recommendation
        Be specific, not generic. No fluff.""",
        task=question
    )
    return {"question": question, "findings": result}

start = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=len(sub_questions)) as executor:
    futures = [executor.submit(research_question, q) for q in sub_questions]
    all_findings = [f.result() for f in concurrent.futures.as_completed(futures)]

print(f"\n⚡ Research completed in {time.time()-start:.1f}s")

# ── STEP 3: Synthesizer combines findings ──
print("\n📊 STEP 3: Synthesizer writing report...")

findings_text = "\n\n".join(
    f"Q: {f['question']}\nA: {f['findings']}" for f in all_findings
)

report = run_agent(
    name="Synthesizer",
    system_prompt="""You are a technical report writer. Given research Q&A findings, 
    synthesize them into a well-structured short report with:
    1. Executive Summary (2 sentences)
    2. Key Findings (one section per sub-topic)
    3. Top 3 Actionable Recommendations
    Use clear headings.""",
    task=f"Topic: {RESEARCH_TOPIC}\n\nResearch Findings:\n{findings_text}"
)

# ── STEP 4: Critic validates the report ──
print("\n🔍 STEP 4: Critic validating report...")

validation = run_agent(
    name="Validator",
    system_prompt="""You are a quality assurance reviewer for technical reports. 
    Review the given report and rate it 1-10 on: clarity, actionability, and completeness.
    Give one sentence of feedback for each dimension. 
    End with: QUALITY SCORE: X/10""",
    task=f"Validate this report:\n{report}"
)

print("\n" + "=" * 60)
print("📄 FINAL REPORT")
print("=" * 60)
print(report)
print("\n" + "─" * 60)
print("🎯 QUALITY VALIDATION")
print("─" * 60)
print(validation)

---
## Part 7: Handoff Patterns — How Agents Pass Work

We've built the mechanics. Now let's think about the *interfaces* between agents — the "contracts" for passing work.

### Three handoff styles:

**1. Direct string pass (what we've done)**
```python
result_a = agent_a(task)
result_b = agent_b(result_a)  # just a string
```
Simple, but fragile — the receiving agent has to parse unstructured text.

**2. Structured JSON handoff**
```python
result_a = agent_a(task)  # returns JSON: {"summary": ..., "confidence": 0.9, "next_steps": [...]}
result_b = agent_b(result_a["summary"])  # consume specific fields
```
More robust — the contract is explicit.

**3. Shared state / message queue**
```python
queue.put({"type": "research_done", "payload": result, "agent_id": "researcher-1"})
# orchestrator reads from queue, routes to next agent
```
Production-grade — decouples agents, enables retry, observability.

Let's implement the structured JSON handoff:

In [ ]:
# ─────────────────────────────────────────────
# STRUCTURED JSON HANDOFF
# Each agent returns a typed dict — the next agent 
# consumes specific fields, not raw text
# ─────────────────────────────────────────────

from dataclasses import dataclass

@dataclass
class AnalysisResult:
    """Structured contract between agents."""
    topic: str
    key_points: list[str]
    confidence: str  # 'high', 'medium', 'low'
    needs_review: bool
    raw_analysis: str

def analyst_agent(topic: str) -> AnalysisResult:
    """Agent 1: Analyzes a topic, returns structured result."""
    
    response = run_agent(
        name="Analyst",
        system_prompt="""You are a technical analyst. Analyze the given topic and respond in EXACTLY this JSON format:
{
  "key_points": ["point 1", "point 2", "point 3"],
  "confidence": "high" or "medium" or "low",
  "needs_review": true or false
}
Output ONLY the JSON, nothing else.""",
        task=f"Analyze: {topic}"
    )
    
    # Parse the structured response
    try:
        match = re.search(r'\{.*?\}', response, re.DOTALL)
        data = json.loads(match.group() if match else response)
    except:
        data = {"key_points": [response], "confidence": "medium", "needs_review": True}
    
    return AnalysisResult(
        topic=topic,
        key_points=data.get("key_points", []),
        confidence=data.get("confidence", "medium"),
        needs_review=data.get("needs_review", False),
        raw_analysis=response
    )

def action_agent(analysis: AnalysisResult) -> str:
    """Agent 2: Takes structured analysis, produces action plan.
    It consumes specific FIELDS, not raw text."""
    
    # Consume structured fields, not raw text!
    context = f"""Topic: {analysis.topic}
Confidence level: {analysis.confidence}
Key findings:
" + "\n".join(f"- {p}" for p in analysis.key_points)
Requires expert review: {analysis.needs_review}"""
    
    return run_agent(
        name="Action Planner",
        system_prompt="Given a structured analysis, produce 3 concrete next actions. If confidence is low or review is needed, make reviewing the first action.",
        task=context
    )

# Run the structured handoff pipeline
print("Running structured handoff pipeline...")
print("=" * 60)

topic = "The security implications of giving AI agents write access to production databases"

# Agent 1 → structured output
analysis = analyst_agent(topic)
print(f"\n📊 Analysis Result (structured):")
print(f"  Topic: {analysis.topic[:60]}...")
print(f"  Key points: {len(analysis.key_points)} found")
print(f"  Confidence: {analysis.confidence}")
print(f"  Needs review: {analysis.needs_review}")

# Agent 2 ← consumes structured fields
action_plan = action_agent(analysis)

print("\n" + "=" * 60)
print("✅ ACTION PLAN (from structured handoff)")
print("=" * 60)
print(action_plan)

---
## Summary: What You Built Today

| Pattern | What It Does | When to Use |
|---|---|---|
| **Sequential Pipeline** | A→B→C chain, each builds on last | Document generation, step-by-step workflows |
| **Orchestrator + Subagents** | Central brain with specialist workers | Complex tasks needing dynamic routing |
| **Parallel Fan-Out** | Many agents run simultaneously | Speed, multiple perspectives, batch processing |
| **Critic/Reviewer Loop** | Generate → critique → revise | Quality-critical outputs (code, writing, plans) |
| **Structured Handoff** | Typed contracts between agents | Production systems needing reliability |

---

## The Big Mental Shift

Single agents think linearly. Multi-agent systems think *organizationally*.

When you design a multi-agent system, you're not just writing prompts — you're designing **roles**, **interfaces**, and **coordination protocols**. It's software architecture, not just prompt engineering.

As a Java developer, you'll recognize this: it's basically microservices for AI. Each agent is a service with a clear responsibility and API contract.

---

## What's Next: Lesson 7 — RAG (Retrieval-Augmented Generation)

Your agents so far only know what's in their context window. In the next lesson, you'll teach them to **retrieve information** — giving them access to your own documents, databases, and knowledge bases.

You'll build:
- **Embeddings**: turning text into semantic vectors
- **Vector search**: finding relevant content by meaning, not keywords
- **A RAG pipeline**: an agent that answers questions using your own documents

This is when agents stop being toys and start being genuinely useful for real knowledge work. 🚀

---
*Lesson 6 of 9 | Learn AI with Gourav | Generated 2026-05-05*

In [ ]:
# 🏆 CHALLENGE: Build Your Own Mini Multi-Agent System
#
# Design a 3-agent system for a problem you care about.
# Ideas:
#   - Code review pipeline: Generator → Security Reviewer → Performance Reviewer → Synthesizer
#   - Interview prep: Topic Planner → Question Generator → Answer Critic
#   - Resume analyzer: Parser → Skills Extractor → Gap Analyzer → Recommendation Writer
#
# Template to get you started:

def my_agent_1(input_text: str) -> str:
    return run_agent(
        name="Agent 1",
        system_prompt="YOUR SYSTEM PROMPT HERE",
        task=input_text
    )

def my_agent_2(input_text: str) -> str:
    return run_agent(
        name="Agent 2",
        system_prompt="YOUR SYSTEM PROMPT HERE",
        task=input_text
    )

def my_agent_3(input_text: str) -> str:
    return run_agent(
        name="Agent 3",
        system_prompt="YOUR SYSTEM PROMPT HERE",
        task=input_text
    )

# Wire them together:
# my_input = "YOUR INPUT HERE"
# step1 = my_agent_1(my_input)
# step2 = my_agent_2(step1)
# step3 = my_agent_3(step2)
# print(step3)

print("Challenge template ready — fill in your system prompts and run!")